# 🕸️ Graph RAG — 금융·하이브리드·법률

> **통합 실습 노트북** — 원본 3개 실습을 하나로 묶었습니다.

## 포함 파트
1. **Lab 5 — Graph RAG (금융 도메인, 메인 트랙)**  (`lab5_graphrag_finance`)
2. **Lab 5b — Hybrid Graph + Vector RAG (그래프·벡터 결합 심화)**  (`lab5b_graphrag_hybrid`)
3. **Lab 6 — Graph RAG (법률 도메인, 확장/도전 트랙)**  (`lab6_graphrag_legal`)

> 🔑 **환경 셋업은 바로 아래 셀에서 한 번만 실행**하면 됩니다(키 입력 1회). 각 파트는 순서대로 이어집니다.

## 🧭 환경 셋업 (맨 처음 한 번만 실행)

In [ ]:
# === 환경 셋업 — 이 통합 노트북에서 "한 번만" 실행하면 전체 파트에서 그대로 씁니다 ===
# 비-Claude 라이브러리는 검색·그래프 인프라용입니다(Claude는 임베딩·BM25·리랭크·그래프 미제공).
!pip install -q "anthropic>=0.40" networkx matplotlib sentence-transformers faiss-cpu httpx

import os, getpass, json, re, math, random, collections, textwrap
if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API Key: ")
from anthropic import Anthropic
client = Anthropic()
MODEL = "claude-sonnet-4-6"   # 정확도가 중요한 judge·추출 단계는 "claude-opus-4-8"로 상향 가능
print("✅ 환경 셋업 완료 — API 키 입력됨, client·MODEL 준비됨")

In [ ]:
# 한글 폰트 설정 (Colab) — 그래프/시각화에서 한글 깨짐 방지
!apt-get install -y fonts-nanum > /dev/null 2>&1
import matplotlib.pyplot as plt, matplotlib.font_manager as fm
_fp = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
try:
    fm.fontManager.addfont(_fp); plt.rcParams['font.family'] = 'NanumGothic'
except Exception:
    pass
plt.rcParams['axes.unicode_minus'] = False
print('한글 폰트 설정 완료')

<!-- V2-MODELNOTE -->
## 🧭 현행 모델 라인업 & 최신 API 옵션 (2026 기준)
이 실습은 **`claude-sonnet-4-6`** 으로 고정합니다(비용·속도·안정성 균형). 아래는 참고용 라인업이며,
*judge·트리플 추출처럼 정확도가 결과를 좌우하는 단계*에서는 셀 주석대로 **`claude-opus-4-8`** 로 한 줄만 올리면 됩니다.

| 모델 | API ID | 컨텍스트 | 가격(입력/출력, /MTok) | 자리매김 |
|---|---|---:|---|---|
| Fable 5 | `claude-fable-5` | 1M | $10 / $50 | 최상위·최난도 장기 에이전트 |
| Opus 4.8 | `claude-opus-4-8` | 1M | $5 / $25 | 어려운 judge·트리플 추출 상향용 |
| **Sonnet 4.6** | `claude-sonnet-4-6` | 1M | $3 / $15 | **실습 기본 — 본 과정 전 노트북 고정** |
| Haiku 4.5 | `claude-haiku-4-5` | 200K | $1 / $5 | 최저가·대량 호출 |

**선택 가이드**: 대량 생성·요약·Self-Query → Sonnet 4.6/Haiku 4.5 · 어려운 judge·추출 → **Opus 4.8로 상향** · 최난도 장기 에이전트 → Fable 5.

**최신 API 옵션(노트)**
- **prompt caching**: 안 바뀌는 앞부분(rubric·context·문서 전문)을 캐시 → 반복 호출 비용 최대 90%↓. (`cache_control: {"type":"ephemeral"}`)
- **Batches API**: 실시간 불필요한 대량 채점은 묶어서 50% 할인(비동기, 최대 24h).
- **adaptive thinking**: 추론 토큰은 `thinking={"type":"adaptive"}` + `output_config={"effort":...}` 사용. ⚠️ 구식 `budget_tokens`는 최신 모델에서 deprecated/오류 → 쓰지 않습니다.
- **structured outputs**: 기존 `tool_use` 강제도 유효하고, 신규 코드는 `output_config={"format":{"type":"json_schema",...}}` 또는 `client.messages.parse(output_format=PydanticModel)`도 가능.

> 💡 모델명만 바꾸면 됩니다: 실제 가용 모델이 다르면 셋업 셀의 `MODEL = ...` 한 줄만 교체하세요.


---
# 파트 1 · Lab 5 — Graph RAG (금융 도메인, 메인 트랙)

## 학습목표
- 금융 도메인에서 **지식 그래프(KG)를 처음부터 직접 구축·질의**한다.
- 벡터 RAG가 못 푸는 **multi-hop / 전역(global) 질문**을 그래프로 해결한다.
- 파이프라인: **벡터 실패 관찰 → 추출 → 정규화 → DiGraph 구축·시각화 → Local search → Global search → 벡터 vs 그래프 정량 비교(judge·시각화) → 중심성 분석**.

## 사전개념
- **트리플(triple)**: (head, relation, tail) — "노바테크 -투자-> 그린모빌리티". 그래프의 최소 단위.
- **DiGraph(방향 그래프)**: 관계의 방향이 의미를 가진다(투자한 쪽 ≠ 받은 쪽).
- **Local search**: 질문 엔티티 주변 **k-hop 서브그래프**로 답한다(구체적 multi-hop).
- **Global search**: 그래프를 커뮤니티로 나눠 각각 요약 후 **map-reduce**로 종합(전역 질문).
- **중심성(centrality)**: degree/PageRank로 "허브 엔티티"를 식별 — 벡터 검색엔 없는 그래프 고유 분석.

> "벡터는 유사한 청크를, 그래프는 연결된 사실을 검색한다(보완재)."
> 비-Claude 사유: networkx(그래프 — Neo4j는 Colab에서 무거워 networkx로 개념 전달, 운영은 Neo4j), matplotlib(시각화), sentence-transformers/faiss(벡터 비교군 — Claude는 임베딩 미제공).

## 노트북 의존
- lab0의 벡터 RAG 패턴을 비교군으로 재사용한다(여기선 자체 정의).
- 이 노트북의 함수(`extract_triples`,`normalize`,`subgraph_text`,`local_search`,`global_search`)는 **lab5b(하이브리드)·lab6(법률)에서 재사용**된다.

## 📖 용어 미니 사전 (먼저 읽으세요)

> 이 실습에 처음 나오는 말들입니다. **뜻만 한 줄로** 잡고 시작하면 코드가 훨씬 쉽게 읽힙니다. 외울 필요 없이, 막힐 때 여기로 돌아오세요.

| 용어 | 한 줄 뜻 |
|---|---|
| **지식 그래프(Knowledge Graph)** | 사실을 "점과 선"으로 그린 지도. 점=대상, 선=관계. |
| **노드(node)** | 그래프의 **점**. 여기선 기업이나 사람(예: 노바테크, 김하늘). |
| **엣지(edge)** | 노드를 잇는 **선** = 관계(예: 노바테크 —투자→ 그린모빌리티). |
| **엔티티(entity) / 관계(relation)** | 엔티티=등장인물(기업·사람), 관계=둘을 잇는 동사(투자/공급/CEO 등). |
| **트리플(triple)** | (주어, 관계, 목적어) 한 묶음. "노바테크 - 투자 - 그린모빌리티". 그래프의 최소 조각. |
| **multi-hop(멀티홉)** | 답을 찾으려 **선을 두 번 이상 갈아타야** 하는 질문. "A에 투자한 회사의 *CEO*는?" → 투자 선 1번 + CEO 선 1번. |
| **서브그래프(subgraph)** | 전체 지도에서 질문과 **관련된 부분만 오려낸 작은 지도**. |
| **Local search** | 질문에 나온 점 **주변만** 보고 답하기(구체적 질문에 강함). |
| **Global search** | 지도를 동네별로 나눠 **각각 요약한 뒤 종합**하기(전체 흐름 질문에 강함). |
| **커뮤니티(community)** | 서로 빽빽하게 연결된 점들의 **무리**(= 동네). |
| **중심성 / PageRank** | 누가 **가장 중심(허브)인가**를 점수로 매기는 법. PageRank는 "중요한 점에게 많이 연결될수록 높은 점수". |
| **networkx** | 그래프를 **만들고 탐색**하는 파이썬 도구(라이브러리). |
| **matplotlib** | 그래프를 **그림으로 그려주는** 도구. |
| **임베딩 / 벡터 검색** | 글을 숫자(벡터)로 바꿔 **의미가 비슷한 글**을 찾는 기존 방식. 이 실습의 "비교 대상". |

## 🧭 왜 "그래프"가 필요할까? (일상 비유로)

전화번호부에서 "**그린모빌리티에 투자한 회사의 CEO 이름**"을 찾는다고 해봅시다.
- 전화번호부(=벡터 검색)는 "그린모빌리티"가 적힌 페이지는 잘 찾습니다. 하지만 **"투자한 회사가 누구인지 → 그 회사의 CEO가 누구인지"** 두 단계를 *연결*해 주지는 못합니다. 답에 필요한 두 사실이 서로 다른 페이지에 흩어져 있으니까요.
- 반면 **인맥 지도**(=지식 그래프)가 있으면, "그린모빌리티" 점에서 **투자 선을 거꾸로 따라가** 투자한 회사를 찾고, 다시 그 회사에서 **CEO 선을 따라가** 사람 이름에 도착합니다. 선을 **갈아타며 따라가는 것**, 이게 그래프의 힘입니다.

이렇게 **관계를 두 번 이상 따라가야 답이 나오는 질문**(multi-hop)이나 **"이 바닥 전체 흐름은?"** 같은 큰 질문은, 키워드가 비슷한 문서를 찾는 것만으로는 풀기 어렵습니다. 그래서 사실들을 **점(기업·사람)과 선(관계)으로 미리 엮은 지도** = 지식 그래프가 필요합니다. 이번 실습에서 그 지도를 직접 만들어 봅니다.

## 데이터 · 가상 금융 공시 (합성·내장, 규모 확대)

비교가 의미 있도록 **가상 기업 13곳 + 임원 + 투자/자회사/경쟁/공급/협력 관계**를 담은 문단형 공시 14건을 정의한다. 실제 기업·사실이 아니다.

### 📌 이 셀이 하는 일 — 가상 데이터 준비
- 실험용 **가짜 금융 공시 14건**을 변수에 담습니다. 실제 기업·사실이 아닙니다(합성 데이터).
- 각 문장에는 "누가-무엇을-누구에게"(투자/공급/CEO 등) 관계가 들어 있습니다. 이게 나중에 **그래프의 재료**가 됩니다.
- 출력 `14 건 공시 로드`는 14개 문장이 잘 담겼다는 뜻입니다.

In [ ]:
DISCLOSURES = [
    "노바테크(NovaTech)는 가상 반도체 설계 기업으로 CEO는 김하늘이다. 노바테크는 2021년 그린모빌리티에 500억 원을 투자했다.",
    "그린모빌리티(GreenMobility)는 전기차 배터리를 만드는 가상 기업이며 CFO는 박서준이다. 그린모빌리티는 오션로지스틱스와 원자재 운송 공급 계약을 맺었다.",
    "오션로지스틱스(OceanLogistics)는 가상 해운 물류 기업으로 CEO는 이도윤이다. 그린모빌리티의 배터리 원자재를 운송한다.",
    "노바테크는 퀀텀칩(QuantumChip)과 AI 가속기 시장에서 경쟁한다. 퀀텀칩의 CEO는 최유나이다.",
    "노바소프트(NovaSoft)는 노바테크의 자회사이며 클라우드 소프트웨어를 만든다. 노바소프트의 대표는 정민수다.",
    "정밀소재(PrecisionMat)는 노바테크와 퀀텀칩 양쪽에 핵심 소재를 공급하는 가상 기업이다.",
    "그린모빌리티는 2024년 IPO를 준비 중이며 노바테크가 주요 투자자로 남아 있다.",
    "퀀텀칩은 정밀소재로부터 핵심 소재를 공급받아 데이터센터용 칩을 생산한다.",
    "스카이센서(SkySensor)는 가상 센서 기업으로 CEO는 한지우다. 스카이센서는 그린모빌리티에 배터리 센서를 공급한다.",
    "딥비전(DeepVision)은 가상 AI 비전 스타트업으로 노바테크가 2023년 200억 원을 투자했다. 딥비전 대표는 윤소라이다.",
    "퀀텀칩은 클라우드넷(CloudNet)과 데이터센터 인프라 협력 관계를 맺었다. 클라우드넷 CEO는 오재성이다.",
    "노바소프트는 클라우드넷의 클라우드 인프라를 사용하며 기술 협력 관계다.",
    "메가뱅크(MegaBank)는 가상 금융사로 노바테크와 그린모빌리티 양쪽에 자금을 대출한 채권자다. 메가뱅크 CEO는 강도현이다.",
    "정밀소재는 스카이센서에도 소재를 공급하며, 딥비전과는 AI 품질검사 기술 협력을 한다.",
]
print(len(DISCLOSURES), "건 공시 로드")

## 1단계 · 왜 그래프인가? (벡터 RAG의 multi-hop 실패 관찰)

### 개념 복습
벡터 RAG는 질문과 **의미가 비슷한 청크 top-k**를 가져온다. 그런데 multi-hop 질문("A에 투자한 회사의 CEO는?")은 정답에 필요한 두 사실(투자 관계 / CEO)이 **서로 다른 문서에 흩어져** 있어, top-k에 둘 다 잡히지 않으면 답을 못 한다. 먼저 이 실패를 눈으로 확인한다.

### 📌 이 셀이 하는 일 — 기존 방식(벡터 검색)의 한계를 눈으로 보기
- 공시 문장들을 숫자로 바꿔(임베딩) **의미가 비슷한 문장을 찾는** 기존 검색기를 만듭니다.
- 그다음 multi-hop 질문 "그린모빌리티에 **투자한 회사의 CEO**는?"을 던집니다.
- **출력 읽는 법**: `[검색된 청크]`는 검색기가 가져온 문장들입니다. 답에 필요한 두 사실(투자 관계 / CEO)이 **여기 함께 잡히지 않으면** 아래 `[벡터 RAG 답변]`이 흔들립니다 — 바로 이 "실패"를 확인하는 게 목적입니다.
> 참고: 임베딩·FAISS는 Claude가 제공하지 않는 기능이라 다른 라이브러리(sentence-transformers/faiss)를 씁니다. 비교 대상일 뿐입니다.

In [ ]:
import numpy as np, faiss
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")  # 임베딩=비-Claude
def embed(t): return np.array(embedder.encode(t, normalize_embeddings=True), dtype="float32")
VEMB = embed(DISCLOSURES); vindex = faiss.IndexFlatIP(VEMB.shape[1]); vindex.add(VEMB)

def vector_chunks(question, k=3):
    _, idx = vindex.search(embed([question]), k)
    return [DISCLOSURES[j] for j in idx[0]]
def vector_rag(question, k=3):
    ctx = "\n".join(f"[{i+1}] {c}" for i,c in enumerate(vector_chunks(question,k)))
    return ask_claude(f"근거:\n{ctx}\n\n질문:{question}\n근거만 사용해 답하라.")

MULTIHOP_Q = "그린모빌리티에 투자한 회사의 CEO는 누구인가?"
print("[검색된 청크]")
for c in vector_chunks(MULTIHOP_Q): print("  -", c[:50], "...")
print("\n[벡터 RAG 답변]", vector_rag(MULTIHOP_Q))
# 관찰: '투자 관계'와 'CEO'가 다른 문서라 top-k에 함께 안 잡히면 답이 흔들린다.

## 2단계 · 엔티티/관계 추출 (Claude tool use)

### 개념 복습
KG의 첫 단계는 비정형 텍스트를 **트리플(head, relation, tail)**로 구조화하는 것이다. Claude의 **tool use(`tool_choice` 강제)**로 항상 JSON 트리플을 받아 파싱을 안정화한다.

### 📌 이 셀이 하는 일 — 문장에서 "관계 조각(트리플)" 뽑아내기
- Claude에게 각 공시 문장을 읽혀 **(주어 - 관계 - 목적어)** 형태의 트리플로 정리하게 합니다. 예: `노바테크 - 투자 -> 그린모빌리티`.
- `tool use`(도구 강제)는 Claude가 **항상 정해진 표 형식(JSON)**으로 답하게 만드는 장치입니다 — 그래야 코드가 안정적으로 받아 처리할 수 있어요.
- **출력 읽는 법**: `추출 트리플 N개`와 샘플 목록이 보입니다. 이 조각들이 곧 그래프의 **점과 선**이 됩니다.

In [ ]:
# 정확도가 더 필요하면 이 judge/추출 셀의 모델을 MODEL_JUDGE = "claude-opus-4-8" 로 한 줄 상향 가능(기본은 비용 위해 sonnet 4.6).
TRIPLE_TOOL = [{"name":"emit_triples","description":"텍스트에서 (head, relation, tail) 트리플 추출",
  "input_schema":{"type":"object","properties":{"triples":{"type":"array","items":{"type":"object",
    "properties":{"head":{"type":"string"},"relation":{"type":"string"},"tail":{"type":"string"}},
    "required":["head","relation","tail"]}}},"required":["triples"]}}]

def extract_triples(text):
    msg = client.messages.create(model=MODEL, max_tokens=1500,
        tools=TRIPLE_TOOL, tool_choice={"type":"tool","name":"emit_triples"},
        messages=[{"role":"user","content":
            "다음 금융 공시에서 기업/인물 간 관계 트리플을 추출하라. "
            "relation은 투자/자회사/경쟁/공급/협력/채권/CEO/CFO/대표 등 간결한 한국어로.\n"+text}])
    for b in msg.content:
        if b.type=="tool_use": return b.input["triples"]
    return []

# 예상 호출: 공시 건수(14). 소규모라 비용 미미.
all_triples = []
for d in DISCLOSURES:
    all_triples += extract_triples(d)
print(f"추출 트리플 {len(all_triples)}개 (샘플 10):")
for t in all_triples[:10]:
    print("  ", t["head"], "-", t["relation"], "->", t["tail"])

## 3단계 · 엔티티 정규화 (동의어·표기 병합)

### 개념 복습
같은 엔티티가 "노바테크" / "노바테크(NovaTech)" / "NovaTech"처럼 다르게 추출되면 그래프가 **쪼개진다(노드 분열)**. 정규화는 그래프 품질의 핵심이다. 여기선 괄호 제거 + 별칭 사전으로 병합한다.

### 📌 이 셀이 하는 일 — 같은 대상의 다른 이름을 하나로 합치기(정규화)
- "노바테크" / "노바테크(NovaTech)" / "NovaTech"가 따로 추출되면 **같은 회사가 세 점으로 쪼개져** 그래프가 망가집니다.
- 이 셀은 괄호를 떼고 별칭표로 묶어 **하나의 이름으로 통일**합니다.
- **출력 읽는 법**: `정규화 전 N -> 후 M`에서 숫자가 줄었다면, 흩어졌던 이름들이 잘 **합쳐졌다**는 뜻입니다.

In [ ]:
ALIASES = {
    "NovaTech":"노바테크","GreenMobility":"그린모빌리티","OceanLogistics":"오션로지스틱스",
    "QuantumChip":"퀀텀칩","NovaSoft":"노바소프트","PrecisionMat":"정밀소재",
    "SkySensor":"스카이센서","DeepVision":"딥비전","CloudNet":"클라우드넷","MegaBank":"메가뱅크",
}
def normalize(name):
    n = re.sub(r"\(.*?\)","",name).strip()      # "노바테크(NovaTech)" -> "노바테크"
    return ALIASES.get(n, n)

before = len({t["head"] for t in all_triples} | {t["tail"] for t in all_triples})
for t in all_triples:
    t["head"] = normalize(t["head"]); t["tail"] = normalize(t["tail"])
after = len({t["head"] for t in all_triples} | {t["tail"] for t in all_triples})
print(f"엔티티 수: 정규화 전 {before} -> 후 {after} (병합 효과)")
print("정규화 후 엔티티 예:", sorted({t["head"] for t in all_triples})[:10])

## 4단계 · networkx DiGraph 구축 + 시각화

### 개념 복습
트리플을 방향 그래프에 적재한다. 엣지에 `relation` 속성을 달아 "무슨 관계인지"를 보존한다. 시각화로 허브/고립 노드를 눈으로 확인한다.

### 📌 이 셀이 하는 일 — 트리플로 진짜 "지도"(그래프) 만들고 그림 그리기
- 앞서 만든 트리플을 `networkx`의 **방향 그래프(DiGraph)**에 하나씩 넣습니다. 방향이 있어 "투자한 쪽 ≠ 받은 쪽"이 구분됩니다.
- `matplotlib`로 그래프를 **그림으로** 그립니다.
- **🖼 그림에서 볼 점**: 동그라미=기업/사람(노드), 화살표=관계(엣지), 화살표 위 글자=관계 종류(투자/공급 등). **선이 많이 모인 동그라미**(허브)와 **외딴 동그라미**를 눈으로 찾아보세요. 배치는 매번 조금 달라질 수 있습니다(정상).

In [ ]:
import networkx as nx, matplotlib.pyplot as plt
G = nx.DiGraph()
for t in all_triples:
    G.add_edge(t["head"], t["tail"], relation=t["relation"])
print(f"노드 {G.number_of_nodes()}개, 엣지 {G.number_of_edges()}개")

plt.figure(figsize=(12, 8))
pos = nx.spring_layout(G, seed=42, k=0.9)
nx.draw(G, pos, with_labels=True, node_color="#cfe8ff", node_size=1500, font_size=8)
nx.draw_networkx_edge_labels(G, pos, edge_labels=nx.get_edge_attributes(G,"relation"), font_size=6)
plt.title("Finance Knowledge Graph (synthetic)"); plt.axis("off"); plt.show()

## 5단계 · Local search (엔티티 → k-hop 서브그래프 → Claude)

### 개념 복습
질문의 핵심 엔티티를 찾아 그 주변 **k-hop 이웃 서브그래프**만 텍스트로 직렬화해 Claude에 준다. multi-hop 정답에 필요한 사실들이 한 컨텍스트에 모이므로 벡터가 놓친 답이 풀린다.

### 📌 이 셀이 하는 일 — Local search: 질문 주변 지도만 오려 답하기
- 질문에 나온 점(엔티티)을 찾고, 그 **주변 2단계(2-hop) 이웃만** 작은 지도로 오려냅니다(`subgraph_text`).
- 그 작은 지도를 글로 적어 Claude에게 주고 답을 받습니다(`local_search`).
- **출력 읽는 법**: `[서브그래프]`에 "그린모빌리티 ←투자― 노바테크"와 "노바테크 ―CEO→ 김하늘"이 **함께** 들어오면, 벡터가 놓쳤던 multi-hop 답(`김하늘`)이 이제 나옵니다.

In [ ]:
def find_entities(question):
    # 그래프 노드 중 질문에 등장하는 것을 매칭(간단·결정적). 필요시 Claude 추출로 대체 가능.
    return [n for n in G.nodes if n in question]

def subgraph_text(entities, hops=2):
    nodes = set(entities)
    for e in entities:
        if e in G:
            nodes |= set(nx.single_source_shortest_path_length(G.to_undirected(), e, cutoff=hops).keys())
    return "\n".join(f"{u} -{d['relation']}-> {v}" for u,v,d in G.edges(data=True) if u in nodes and v in nodes)

def local_search(question, hops=2):
    ctx = subgraph_text(find_entities(question), hops) or "관련 관계 없음"
    return ask_claude(f"다음 지식그래프 관계만 사용해 답하라:\n{ctx}\n\n질문:{question}")

print("핵심 엔티티:", find_entities(MULTIHOP_Q))
print("[서브그래프]\n", subgraph_text(find_entities(MULTIHOP_Q)))
print("[Graph Local]", local_search(MULTIHOP_Q))

## 6단계 · Global search (커뮤니티 요약 → map-reduce)

### 개념 복습
전역 질문("이 산업군의 주요 흐름은?")은 특정 엔티티 주변이 아니라 **그래프 전체의 패턴**을 묻는다. 그래프를 커뮤니티(여기선 약한 연결요소)로 나눠 각각 Claude로 **요약(map)**하고, 요약들을 모아 **종합(reduce)**한다. 이것이 GraphRAG의 Global search 골격이다.

### 📌 이 셀이 하는 일 — Global search: 동네별 요약 후 종합하기
- "이 바닥 전체 흐름은?" 같은 큰 질문은 한 점 주변만 봐선 안 됩니다.
- 그래프를 **연결된 무리(커뮤니티)**로 나눠 각각 Claude로 **요약(map)**하고, 그 요약들을 모아 다시 **종합(reduce)**합니다. 이 "나눠 요약 → 합치기"가 Global search의 핵심입니다.
- **출력 읽는 법**: 개별 사실이 아니라 **생태계 전반의 흐름**을 정리한 문단이 나오면 성공입니다.

In [ ]:
def community_summaries():
    comms = list(nx.weakly_connected_components(G))
    summaries = []
    for c in comms:                      # map
        sub = G.subgraph(c)
        rels = "\n".join(f"{u} -{d['relation']}-> {v}" for u,v,d in sub.edges(data=True))
        summaries.append(ask_claude(f"다음 관계 묶음의 핵심을 2문장으로 요약:\n{rels}"))
    return summaries

def global_search(question):
    summ = community_summaries()
    joined = "\n".join(f"[요약{i+1}] {s}" for i,s in enumerate(summ))
    return ask_claude(f"커뮤니티 요약들:\n{joined}\n\n전역 질문:{question}\n요약들을 종합해 답하라.")  # reduce

GLOBAL_Q = "이 기업 생태계 전반의 주요 투자·공급 흐름과 핵심 축은 무엇인가?"
print("[Graph Global]", global_search(GLOBAL_Q))

## 7단계 · 벡터 vs 그래프 정량 비교 (judge 채점 + 시각화)

### 개념 복습
"그래프가 좋다"를 **숫자로** 보인다. multi-hop·전역 질문셋을 두 방식으로 답하고 Claude judge로 **정확성**을 1-5로 채점, 막대그래프로 비교한다.

### 📌 이 셀이 하는 일 — "그래프가 정말 더 나은가?"를 점수로 비교
- 같은 질문 4개를 **벡터 방식**과 **그래프 방식** 둘 다로 답하게 합니다.
- 그다음 Claude를 **채점관(judge)**으로 세워 각 답을 정확성 1~5점으로 매깁니다.
- **출력 읽는 법**: 질문별로 두 방식의 점수가 표로 나옵니다. multi-hop·전역 질문에서 보통 **그래프 점수가 높게** 나옵니다.
> 채점관 점수는 실행마다 약간 달라질 수 있어요. **절대 점수보다 "어느 쪽이 더 높은 경향인가"**를 보세요.

In [ ]:
# 정확도가 더 필요하면 이 judge/추출 셀의 모델을 MODEL_JUDGE = "claude-opus-4-8" 로 한 줄 상향 가능(기본은 비용 위해 sonnet 4.6).
JUDGE_TOOL = [{"name":"judge","description":"답변 채점",
  "input_schema":{"type":"object","properties":{
    "reasoning":{"type":"string"},"correctness":{"type":"integer","description":"1-5 정확성"}},
    "required":["reasoning","correctness"]}}]
def judge_answer(question, answer, gold):
    msg = client.messages.create(model=MODEL, max_tokens=500,
        tools=JUDGE_TOOL, tool_choice={"type":"tool","name":"judge"},
        messages=[{"role":"user","content":
            f"질문:{question}\n참고 정답 단서:{gold}\n답변:{answer}\n정확성 1-5로 채점(reasoning 먼저)."}])
    for b in msg.content:
        if b.type=="tool_use": return b.input.get("correctness", 0) # Safely get correctness with default
    return 0

# multi-hop + 전역 혼합 질문셋(정답 단서 포함). 예상 호출: 4질문 x (2답변+2judge)=16건.
QSET = [
    ("그린모빌리티에 투자한 회사의 CEO는?", "노바테크 CEO 김하늘", "local"),
    ("노바테크와 퀀텀칩이 공유하는 공급사는?", "정밀소재", "local"),
    ("노바테크가 투자한 가상 기업들을 모두 나열하면?", "그린모빌리티, 딥비전", "local"),
    ("이 기업 생태계의 주요 투자·공급 흐름은?", "노바테크 중심 투자, 정밀소재 공급 허브 등", "global"),
]
v_scores, g_scores, labels = [], [], []
for q, gold, mode in QSET:
    v = vector_rag(q)
    g = global_search(q) if mode=="global" else local_search(q)
    v_scores.append(judge_answer(q, v, gold)); g_scores.append(judge_answer(q, g, gold))
    labels.append(q[:14])
print(f"{'질문':<16}{'벡터':<6}{'그래프'}")
for l,v,g in zip(labels, v_scores, g_scores):
    print(f"{l:<16}{v:<6}{g}")

### 📌 이 셀이 하는 일 — 비교 결과를 막대그래프로 그리기
- 위에서 매긴 점수를 **막대그래프**로 그려 한눈에 비교합니다.
- **🖼 그림에서 볼 점**: 질문마다 막대 2개(벡터 vs 그래프)가 나란히 섭니다. **그래프 막대가 더 높은 질문**이 "그래프라서 풀린" 질문입니다. 맨 아래 줄에 **평균 점수**도 찍힙니다.

In [ ]:
x = np.arange(len(labels)); w = 0.38
plt.figure(figsize=(9,4))
plt.bar(x-w/2, v_scores, w, label="벡터 RAG")
plt.bar(x+w/2, g_scores, w, label="Graph RAG")
plt.xticks(x, labels, rotation=15); plt.ylim(0,5.3); plt.ylabel("정확성(1-5)")
plt.title("Vector vs Graph RAG — 질문별 judge 정확성"); plt.legend()
plt.tight_layout(); plt.show()
print(f"평균 정확성  벡터={np.mean(v_scores):.2f}  그래프={np.mean(g_scores):.2f}")

### 결과 해설
- multi-hop·전역 질문에서 **Graph RAG가 평균적으로 높다**. 연결을 따라가거나 전체를 요약하기 때문이다.
- 단일 청크로 답 가능한 단순 질문이라면 벡터도 충분하다 → 실무는 **질문 유형에 따라 라우팅**(lab5b 하이브리드로 발전).
- judge는 변동이 있으므로 절대값보다 **경향**을 본다.

## 8단계 · 중심성 분석 (그래프 고유: degree / PageRank)

### 개념 복습
벡터 검색엔 없는 그래프만의 분석이다. **degree(연결 수)**, **PageRank(영향력)**로 생태계의 **허브 기업**을 식별한다 — "누가 가장 중심인가?"는 그래프로만 답할 수 있다.

### 📌 이 셀이 하는 일 — 누가 "허브"인가? (그래프만의 분석: 중심성)
- 벡터 검색으로는 못 하는, **그래프만의 분석**입니다.
- `degree`(연결된 선의 총 개수)와 `PageRank`("중요한 점에게 많이 연결될수록 높은 점수")로 **이 생태계의 중심 기업**을 찾습니다.
- **🖼 그림에서 볼 점**: 동그라미 크기가 PageRank에 비례합니다 — **가장 큰 동그라미가 핵심 허브**(예: 노바테크, 정밀소재)입니다. "누가 가장 중심인가?"는 그래프로만 답할 수 있는 질문입니다.

In [ ]:
deg = dict(G.degree())                 # 총 연결 수
indeg = dict(G.in_degree())            # 피지목(투자받음/공급받음 등 대상이 되는 정도)
pr = nx.pagerank(G)
top_deg = sorted(deg.items(), key=lambda x:x[1], reverse=True)[:5]
top_pr  = sorted(pr.items(),  key=lambda x:x[1], reverse=True)[:5]
print("degree 상위:", top_deg)
print("PageRank 상위:", [(k, round(v,3)) for k,v in top_pr])

# 노드 크기를 PageRank에 비례시켜 허브를 시각적으로 강조
plt.figure(figsize=(11,7))
pos = nx.spring_layout(G, seed=42, k=0.9)
sizes = [4000*pr[n] + 300 for n in G.nodes]
nx.draw(G, pos, with_labels=True, node_color="#ffd9b3", node_size=sizes, font_size=8)
plt.title("PageRank 기반 허브 강조 (노드 크기 ∝ PageRank)"); plt.axis("off"); plt.show()
print("\n해석: PageRank/degree 상위 기업이 이 생태계의 핵심 허브(예: 노바테크, 정밀소재).")

## 9단계 · 🔧 직접 해보기 (3개 이상)

In [ ]:
# 🔧 직접 해보기 — 아래 3개 중 하나씩 골라, 주석(#)을 지우고 빈칸을 채워 실행하세요.
# 막히면 "힌트"의 예시를 거의 그대로 따라 해도 됩니다. 정답이 하나만 있는 건 아니에요.

# ───────────────────────────────────────────────────────────────
# 🔧 직접 1) 새 관계를 그래프에 추가하고 multi-hop 질문 던져보기
#   힌트: G.add_edge("출발노드", "도착노드", relation="관계이름") 형태입니다.
#   예시(아래 두 줄의 # 을 지우면 바로 동작):
# G.add_edge("딥비전", "스카이센서", relation="협력")
# print(local_search("노바테크가 투자한 회사와 협력하는 센서 기업은?"))

# ───────────────────────────────────────────────────────────────
# 🔧 직접 2) "몇 단계까지 이웃을 볼지(hops)"를 바꿔 서브그래프 크기 비교하기
#   힌트: hops 숫자가 커질수록 더 넓은 지도를 봅니다. 1, 2, 3을 비교해 보세요.
# for h in (1, 2, 3):
#     print(f"hops={h} → 서브그래프 글자 수:", len(subgraph_text(["그린모빌리티"], hops=h)))

# ───────────────────────────────────────────────────────────────
# 🔧 직접 3) 질문 종류에 따라 Local/Global을 자동으로 고르는 규칙 만들기
#   힌트: 질문에 "전반/생태계/흐름" 같은 큰 단어가 있으면 global, 아니면 local 로 보냅니다.
# def route(q):
#     return "global" if any(w in q for w in ["전반", "생태계", "흐름", "경향", "핵심 축"]) else "local"
# def smart_answer(q):
#     return global_search(q) if route(q) == "global" else local_search(q)
# print(route("이 생태계의 핵심 축은?"), "/", route("그린모빌리티 투자자의 CEO는?"))

print("위 예시 중 하나를 골라 # 을 지우고 실행해보세요. 잘 안 되면 바로 위 셀들의 함수 이름을 참고하세요.")


## 10단계 · ✅ 검증

### 📌 이 셀이 하는 일 — 자동 점검(검증)
- 그래프가 충분히 만들어졌는지, multi-hop 정답(`김하늘`)이 그래프 답에 들어 있는지 등을 `assert`로 **자동 확인**합니다.
- `assert`는 "이 조건이 참이어야 한다"는 검사입니다. 조건이 깨지면 빨간 오류가 나고, 통과하면 마지막 줄 `✅ ... 검증 통과`가 찍힙니다.
- ⚠️ 여기서 오류가 나면, 앞 단계(추출/정규화/그래프 구축) 셀을 위에서부터 다시 실행했는지 확인하세요.

In [ ]:
assert G.number_of_nodes() >= 12, f"엔티티가 적음({G.number_of_nodes()}) — 추출/정규화 확인"
assert G.number_of_edges() >= 12, f"관계가 적음({G.number_of_edges()}) — 추출 확인"
# multi-hop: 그린모빌리티 투자자(노바테크)와 CEO(김하늘)가 2-hop 서브그래프 안에 있어야
ctx = subgraph_text(["그린모빌리티"], hops=2)
assert "노바테크" in ctx, "투자자가 서브그래프에 없음"
ans = local_search(MULTIHOP_Q)
assert "김하늘" in ans, f"multi-hop 정답(김하늘)이 그래프 답변에 없음: {ans}"
# 중심성 결과가 비어있지 않아야
assert len(pr) == G.number_of_nodes() and max(pr.values()) > 0, "PageRank 계산 이상"
# 정량 비교에서 그래프 평균이 벡터 이상(경향)
assert np.mean(g_scores) >= np.mean(v_scores) - 0.5, "그래프가 벡터보다 크게 낮음 — 질문셋/검색 점검"
print("✅ lab5 Graph RAG(금융) 검증 통과")

## 확장과제
- **하이브리드 context**(→ lab5b): 그래프 서브그래프 + 벡터 top-k 청크를 함께 Claude에 줘 답변 품질 비교.
- **커뮤니티 탐지 고도화**: 현재 Global search의 단순 그룹 분할을 **진짜 커뮤니티 탐지(`python-louvain`/Louvain, 또는 `networkx`의 greedy_modularity_communities)로 교체**해 의미 있는 커뮤니티로 묶고 요약 품질을 비교한다.
- **Neo4j 마이그레이션(개념)**: 트리플을 Cypher `MERGE (a)-[:REL]->(b)`로 적재하는 맛보기.
- **자동 라우팅**: Local/Global을 Claude 분류기로 판단해 `smart_answer` 완성.

> 이 노트북의 `extract_triples`,`normalize`,`subgraph_text`,`local_search`,`global_search` 패턴은 lab5b(하이브리드)·lab6(법률)에서 재사용된다.

---
# 파트 2 · Lab 5b — Hybrid Graph + Vector RAG (그래프·벡터 결합 심화)

## 학습목표
- lab5에서 만든 **지식 그래프 서브그래프**와 lab0식 **벡터 청크 검색**을 **하나의 컨텍스트로 결합**한다.
- 벡터 / 그래프 / **하이브리드** 세 방식을 같은 질문셋으로 돌려 **judge 정량 비교 + 막대그래프 시각화**한다.
- **시간 추론(temporal reasoning)** 맛보기: 사건 연도를 파싱해 "선후 관계(누가 먼저였나)"를 그래프로 답한다.

## 왜 하이브리드인가? (개념)
| 방식 | 강점 | 약점 |
|---|---|---|
| 벡터 RAG | 의미 유사 청크, 풍부한 서술/맥락 | multi-hop·연결 추적 약함, 흩어진 사실 누락 |
| Graph RAG | 연결된 사실·경로·집계 | 원문 뉘앙스/수치 서술이 트리플로 손실될 수 있음 |
| **Hybrid** | 그래프로 **연결 사실**, 벡터로 **원문 근거** → 보완 | context가 길어짐(비용↑), 융합 설계 필요 |

> "그래프는 *연결*을, 벡터는 *서술*을 가져온다. 둘을 합치면 multi-hop 정답 + 근거 문장을 동시에 얻는다."
> 비-Claude 사유: networkx(그래프 — Neo4j는 Colab에서 무거움), sentence-transformers/faiss(벡터검색 — Claude는 임베딩 미제공), matplotlib(시각화).

## 📖 용어 미니 사전 (먼저 읽으세요)

> lab5(금융 그래프)를 먼저 했다면 익숙한 말이 많습니다. 이번엔 **두 방식을 합치는(하이브리드)** 게 주제예요.

| 용어 | 한 줄 뜻 |
|---|---|
| **벡터 검색(vector)** | 글을 숫자로 바꿔 **의미가 비슷한 원문 문장**을 찾는 방식. 금액·시점 같은 *서술*에 강함. |
| **지식 그래프(graph)** | 사실을 점(기업·사람)과 선(관계)으로 엮은 지도. *연결(누가-무엇을-누구에게)*에 강함. |
| **노드 / 엣지** | 노드=점(기업·사람), 엣지=선(관계). 엣지엔 관계 이름이 붙음(투자/공급 등). |
| **서브그래프** | 질문과 관련된 부분만 오려낸 작은 지도. |
| **하이브리드(hybrid)** | 그래프의 *연결*과 벡터의 *원문 서술*을 **한 컨텍스트로 합쳐** Claude에 주는 방식. |
| **multi-hop(멀티홉)** | 선을 두 번 이상 갈아타야 답이 나오는 질문(예: 투자→CEO). |
| **시간 추론(temporal)** | "무엇이 먼저였나" 같은 **선후 관계**를 연도로 따지는 것. |
| **judge(채점관)** | 답이 얼마나 정확/완전한지 Claude가 1~5점으로 매기게 하는 평가 방식. |
| **컨텍스트(context)** | Claude에게 "이 근거를 보고 답하라"며 같이 넣어주는 참고 자료 묶음. |

## 🧭 왜 "합치는가"? (일상 비유로)

길을 찾을 때 **지하철 노선도**(연결만 또렷)와 **상세 거리 지도**(가게·거리·풍경)를 같이 보면 가장 정확하죠.
- **그래프**는 지하철 노선도 같습니다 — "A에서 B로, 다시 C로 어떻게 이어지는가"(연결)는 분명하지만, *금액·연도 같은 세부 서술*은 빠집니다.
- **벡터 검색**은 상세 지도 같습니다 — "2021년에 500억을 투자" 같은 *원문 문장*은 잘 찾지만, 여러 단계를 *연결*해 따라가긴 어렵습니다.

그래서 이번 실습은 **둘을 한 화면에 겹쳐** Claude에게 줍니다(하이브리드). "연결은 그래프로, 근거 문장은 원문으로" 동시에 활용해 **더 정확하고 빠짐없는 답**을 얻는 게 목표입니다.

## 1단계 · 데이터: 서술이 풍부한 가상 금융 공시 (합성·내장)

하이브리드의 가치를 보이려면 **연도/수치/서술**이 함께 든 문단형 공시가 필요하다. 그래프는 (누가-무엇을-누구에게) 연결을, 벡터는 *금액·시점·맥락 문장*을 책임진다. 실제 기업·사실 아님.

### 📌 이 셀이 하는 일 — 가상 데이터 준비 (서술이 풍부한 공시)
- **연도·금액·맥락**이 한 문장에 함께 든 가짜 공시 8건을 담습니다(실제 아님).
- 하이브리드의 가치를 보이려면 이런 **서술 정보**가 필요합니다 — 벡터가 이 부분을 책임지거든요. `8 건 공시 로드`가 보이면 OK.

In [ ]:
DISCLOSURES = [
    "NovaTech (NovaTech) is a virtual semiconductor design company, and its CEO is Kim Ha-neul. In 2021, it made a primary investment of 50 billion won in GreenMobility, and in 2023, it became the largest shareholder by investing an additional 30 billion won.",
    "GreenMobility (GreenMobility) is a virtual electric vehicle battery company, and its CFO is Park Seo-joon. In 2022, it signed a raw material transportation supply contract with OceanLogistics and is preparing for an IPO in 2024.",
    "OceanLogistics (OceanLogistics) is a virtual maritime logistics company, and its CEO is Lee Do-yoon. Since 2022, it has been exclusively transporting GreenMobility battery raw materials.",
    "NovaTech has been competing with QuantumChip (QuantumChip) in the AI accelerator market since 2020. QuantumChip's CEO is Choi Yu-na, and in 2023, it increased its market share with data center chips.",
    "NovaSoft (NovaSoft) is a subsidiary of NovaTech, established in 2019, that develops cloud software. Its CEO is Jung Min-soo, and it turned profitable in 2023.",
    "PrecisionMat (PrecisionMat) is a virtual company that supplies core materials to both NovaTech and QuantumChip, and in 2021, it signed a long-term supply contract with NovaTech.",
    "QuantumChip produces data center chips by receiving materials from PrecisionMat, and in 2024, it began supplying chips to GreenMobility as well.",
    "Immediately after NovaTech's additional investment in 2023, GreenMobility doubled its production capacity and relies on OceanLogistics' logistics network expansion.",
]
print(len(DISCLOSURES), "disclosures loaded")

## 2단계 · 벡터 인덱스 구축 (lab0 재사용 패턴)

문단형 공시를 임베딩해 FAISS 코사인 인덱스로 만든다. 이것이 **하이브리드의 '서술' 채널**이다.

### 📌 이 셀이 하는 일 — 벡터 채널 만들기 (하이브리드의 '서술' 담당)
- 공시 문장들을 숫자로 바꿔(임베딩) **의미가 비슷한 원문을 찾는** 검색기를 만듭니다.
- 이게 하이브리드의 두 채널 중 **'원문 서술' 채널**입니다. `벡터 인덱스: 8 개 청크`가 나오면 성공.
> 임베딩·FAISS는 Claude가 제공하지 않아 다른 라이브러리(sentence-transformers/faiss)를 씁니다.

In [ ]:
import numpy as np, faiss
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")  # 임베딩=비-Claude(Claude 미제공)
def embed(t): return np.array(embedder.encode(t, normalize_embeddings=True), dtype="float32")

VEMB = embed(DISCLOSURES)
vindex = faiss.IndexFlatIP(VEMB.shape[1]); vindex.add(VEMB)

def vector_chunks(question, k=3):
    _, idx = vindex.search(embed([question]), k)
    return [DISCLOSURES[j] for j in idx[0]]

def vector_rag(question, k=3):
    ctx = "\n".join(f"[{i+1}] {c}" for i, c in enumerate(vector_chunks(question, k)))
    return ask_claude(f"근거:\n{ctx}\n\n질문:{question}\n근거만 사용해 답하라.")

print("벡터 인덱스:", vindex.ntotal, "개 청크")

## 3단계 · 지식 그래프 구축 (Claude 트리플 추출 → DiGraph)

lab5와 동일한 추출·정규화 패턴. 이것이 **하이브리드의 '연결' 채널**이다.

### 📌 이 셀이 하는 일 — 그래프 채널 만들기 (하이브리드의 '연결' 담당)
- lab5와 같은 방식으로 Claude가 문장에서 **트리플(점-관계-점)**을 뽑고, 같은 이름을 합친 뒤(정규화) `networkx` 그래프 `G`에 담습니다.
- 이게 하이브리드의 또 다른 채널인 **'연결' 채널**입니다.
- **출력 읽는 법**: `그래프 노드 N 엣지 M` — 점과 선이 충분히 만들어졌는지 확인합니다.

In [ ]:
# 정확도가 더 필요하면 이 judge/추출 셀의 모델을 MODEL_JUDGE = "claude-opus-4-8" 로 한 줄 상향 가능(기본은 비용 위해 sonnet 4.6).
TRIPLE_TOOL = [{"name":"emit_triples","description":"Extract (head,relation,tail) triples",
  "input_schema":{"type":"object","properties":{"triples":{"type":"array","items":{"type":"object",
    "properties":{"head":{"type":"string"},"relation":{"type":"string"},"tail":{"type":"string"}},
    "required":["head","relation","tail"]}}},"required":["triples"]}}]

def extract_triples(text):
    msg = client.messages.create(model=MODEL, max_tokens=1500,
        tools=TRIPLE_TOOL, tool_choice={"type":"tool","name":"emit_triples"},
        messages=[{"role":"user","content":
            """Extract (head, relation, tail) triples from the following disclosure.
Relations should be concise terms like Investment/Subsidiary/Competition/Supply/CEO/CFO/Representative.
"""+text}])
    for b in msg.content:
        if b.type=="tool_use": return b.input["triples"]
    return []

ALIASES = {"NovaTech":"NovaTech","GreenMobility":"GreenMobility","OceanLogistics":"OceanLogistics",
           "QuantumChip":"QuantumChip","NovaSoft":"NovaSoft","PrecisionMat":"PrecisionMat"}
def normalize(name):
    n = re.sub(r"\(.*?\)","",name).strip()
    return ALIASES.get(n, n)

import networkx as nx, matplotlib.pyplot as plt
all_triples = []
for d in DISCLOSURES:
    all_triples += extract_triples(d)
G = nx.DiGraph()
for t in all_triples:
    h, tl = normalize(t["head"]), normalize(t["tail"])
    G.add_edge(h, tl, relation=t["relation"])
print(f"Graph Nodes {G.number_of_nodes()} Edges {G.number_of_edges()}")

## 4단계 · 그래프 채널: 엔티티 매칭 → k-hop 서브그래프 텍스트화

### 📌 이 셀이 하는 일 — 그래프에서 질문 주변 작은 지도 오려내기
- 질문에 등장하는 점을 찾아(`find_entities`), 그 **주변 이웃만** 글로 정리합니다(`subgraph_text`).
- `graph_rag`는 그 작은 지도만 보고 Claude가 답하게 합니다(그래프 단독 방식).
- **출력 읽는 법**: 아래 출력에 "그린모빌리티 ←투자― 노바테크"처럼 **연결**이 보이면, 그래프 채널이 잘 동작하는 것입니다.

In [ ]:
def find_entities(question):
    return [n for n in G.nodes if n in question]

def subgraph_text(entities, hops=2):
    nodes = set(entities)
    for e in entities:
        if e in G:
            nodes |= set(nx.single_source_shortest_path_length(G.to_undirected(), e, cutoff=hops).keys())
    return "\n".join(f"{u} -{d['relation']}-> {v}" for u,v,d in G.edges(data=True) if u in nodes and v in nodes)

def graph_rag(question, hops=2):
    ctx = subgraph_text(find_entities(question), hops) or "관련 관계 없음"
    return ask_claude(f"다음 지식그래프 관계만 사용해 답하라:\n{ctx}\n\n질문:{question}")

demo_q = "그린모빌리티에 투자한 회사의 CEO는 누구인가?"
print("엔티티:", find_entities(demo_q))
print(subgraph_text(find_entities(demo_q)))

## 5단계 · ⭐ 하이브리드 융합 (그래프 서브그래프 + 벡터 청크)

**핵심 셀.** 그래프 관계(연결)와 벡터 청크(서술)를 **하나의 컨텍스트**로 합쳐 Claude에 전달한다.
- 그래프 채널: multi-hop 경로/연결 사실
- 벡터 채널: 금액·연도·맥락이 든 원문 문장

두 채널을 라벨로 구분해 주면 모델이 "연결은 그래프로, 근거 문장은 벡터로" 활용한다.

### 📌 이 셀이 하는 일 — ⭐핵심: 그래프 + 벡터를 한 컨텍스트로 합치기(하이브리드)
- `hybrid_rag`는 **[그래프 관계]**(연결)와 **[관련 문서]**(원문 서술)를 **하나의 프롬프트로 묶어** Claude에 줍니다.
- 두 채널에 라벨을 붙여주면, Claude가 "연결은 그래프에서, 금액·시점은 문서에서" 골라 씁니다.
- **출력 읽는 법**: `[Hybrid]` 답에 multi-hop 정답(CEO 이름)과 함께 금액·연도 같은 근거가 같이 들어오면 성공입니다.

In [ ]:
def hybrid_rag(question, hops=2, k=3):
    g_ctx = subgraph_text(find_entities(question), hops) or "(관련 그래프 관계 없음)"
    v_ctx = "\n".join(f"- {c}" for c in vector_chunks(question, k))
    prompt = (
        "아래 두 종류의 근거를 함께 사용해 답하라.\n"
        "그래프는 엔티티 간 *연결*을, 문서는 *원문 서술(금액·시점 등)*을 제공한다.\n\n"
        f"[그래프 관계]\n{g_ctx}\n\n[관련 문서]\n{v_ctx}\n\n"
        f"질문: {question}\n근거에 있는 사실만 사용하고, 연결과 서술을 모두 반영해 답하라.")
    return ask_claude(prompt)

print("[Hybrid]", hybrid_rag(demo_q))

## 6단계 · 시간 추론(temporal) 맛보기 — 사건 선후 관계

공시 문장에서 **연도**를 파싱해, "노바테크의 1차 투자와 추가 투자 중 무엇이 먼저인가?" 같은
**선후(before/after) 질문**에 그래프로 답한다. 시간 엣지는 Graph RAG가 흔히 다루는 확장 축이다.

### 📌 이 셀이 하는 일 — 시간 추론 맛보기 ("무엇이 먼저였나")
- 공시 문장에서 **연도(20XX년)**를 정규식으로 뽑아 시간순으로 줄을 세웁니다(`EVENTS`).
- `temporal_answer`는 이 시간표를 근거로 "1차 투자와 추가 투자 중 무엇이 먼저?" 같은 **선후 질문**에 답합니다.
- **출력 읽는 법**: 답이 연도를 근거로 "X가 먼저, Y년 차이"처럼 분명하면 잘 된 것입니다.
> 참고: 정규식(`re.finditer`)은 "20으로 시작하는 4자리 연도"를 글에서 찾아내는 패턴 검색입니다.

In [ ]:
# 사건(이벤트) 노드를 연도와 함께 추출: 간단 규칙 기반 + 그래프에 timeline 부여
EVENTS = []   # (연도, 설명)
for d in DISCLOSURES:
    for m in re.finditer(r"(20\d{2})년[^.]*", d):
        EVENTS.append((int(m.group(1)), m.group(0).strip()))
EVENTS = sorted(set(EVENTS))
TG = nx.DiGraph()                      # temporal chain: 연도 오름차순 연결
prev = None
for yr, desc in EVENTS:
    node = f"{yr}:{desc[:24]}"
    TG.add_node(node, year=yr, desc=desc)
    if prev is not None:
        TG.add_edge(prev, node, relation="이후")   # prev -이후-> node
    prev = node

def temporal_answer(question):
    timeline = "\n".join(f"{yr}년: {desc}" for yr, desc in EVENTS)
    return ask_claude(f"시간순 사건:\n{timeline}\n\n질문:{question}\n연도를 근거로 선후를 분명히 답하라.")

print(f"이벤트 {len(EVENTS)}개 추출")
print("[Temporal]", temporal_answer("노바테크의 그린모빌리티 1차 투자와 추가 투자 중 무엇이 먼저였고 몇 년 차이인가?"))

## 7단계 · ⭐ 벡터 / 그래프 / 하이브리드 3방식 정량 비교 (judge + 시각화)

같은 질문셋을 세 방식으로 답하고, Claude judge로 **정확성·완전성**을 채점한 뒤 막대그래프로 비교한다.

### 📌 이 셀이 하는 일 — 세 방식(벡터/그래프/하이브리드)을 점수로 비교
- 같은 질문 3개를 **세 방식 모두**로 답하게 한 뒤, Claude **채점관(judge)**이 *정확성*과 *완전성*을 1~5점으로 매깁니다.
- **완전성** = 필요한 연결·사실을 빠짐없이 담았는가.
- **출력 읽는 법**: 질문별로 세 방식의 점수가 표로 나옵니다. 보통 **하이브리드가 완전성에서 앞섭니다**.

In [ ]:
# 정확도가 더 필요하면 이 judge/추출 셀의 모델을 MODEL_JUDGE = "claude-opus-4-8" 로 한 줄 상향 가능(기본은 비용 위해 sonnet 4.6).
JUDGE_TOOL = [{"name":"judge","description":"RAG 답변 채점",
  "input_schema":{"type":"object","properties":{
    "reasoning":{"type":"string"},
    "correctness":{"type":"integer","description":"1-5 정확성"},
    "completeness":{"type":"integer","description":"1-5 완전성(필요한 연결/사실을 빠짐없이)"}},
    "required":["reasoning","correctness","completeness"]}}]

def judge(question, answer, gold_hint):
    msg = client.messages.create(model=MODEL, max_tokens=500,
        tools=JUDGE_TOOL, tool_choice={"type":"tool","name":"judge"},
        messages=[{"role":"user","content":
            f"질문:{question}\n참고 정답 단서:{gold_hint}\n답변:{answer}\n"
            "정확성/완전성을 1-5로 채점(reasoning 먼저)."}])
    for b in msg.content:
        if b.type=="tool_use": return b.input
    return {"correctness":0,"completeness":0}

# multi-hop + 시간 + 전역 혼합 질문셋(정답 단서 포함)
QSET = [
    ("그린모빌리티에 투자한 회사의 CEO는?", "노바테크의 CEO 김하늘"),
    ("노바테크가 그린모빌리티에 투자한 총액과 시점은?", "2021년 500억+2023년 300억=800억, 최대주주"),
    ("퀀텀칩과 노바테크가 공유하는 공급사는?", "정밀소재"),
]
# 예상 호출: 3질문 x (3답변 + 3judge) = 18건. 소규모.
import collections
scores = collections.defaultdict(lambda: {"correctness":[], "completeness":[]})
rows = []
for q, gold in QSET:
    answers = {"vector": vector_rag(q), "graph": graph_rag(q), "hybrid": hybrid_rag(q)}
    for method, ans in answers.items():
        j = judge(q, ans, gold)
        scores[method]["correctness"].append(j["correctness"])
        scores[method]["completeness"].append(j["completeness"])
        rows.append((q[:18], method, j["correctness"], j["completeness"]))

print(f"{'질문':<20}{'방식':<8}{'정확성':<7}{'완전성'}")
for q,m,c,comp in rows:
    print(f"{q:<20}{m:<8}{c:<7}{comp}")

### 📌 이 셀이 하는 일 — 비교 결과를 막대그래프로 그리기
- 세 방식의 **평균 정확성·완전성**을 막대그래프로 그립니다.
- **🖼 그림에서 볼 점**: 방식(vector/graph/hybrid)마다 막대 2개(정확성·완전성)가 섭니다. **hybrid 막대가 가장 높은지**, 특히 *완전성*에서 앞서는지 보세요. 막대 위 숫자가 평균 점수입니다.

In [ ]:
# 평균 점수 막대그래프
methods = ["vector","graph","hybrid"]
avg_corr = [np.mean(scores[m]["correctness"]) for m in methods]
avg_comp = [np.mean(scores[m]["completeness"]) for m in methods]
x = np.arange(len(methods)); w = 0.35
plt.figure(figsize=(7,4))
plt.bar(x-w/2, avg_corr, w, label="정확성")
plt.bar(x+w/2, avg_comp, w, label="완전성")
plt.xticks(x, methods); plt.ylim(0,5.2); plt.ylabel("평균 점수(1-5)")
plt.title("Vector vs Graph vs Hybrid (judge 평균)"); plt.legend()
for i,(a,b) in enumerate(zip(avg_corr,avg_comp)):
    plt.text(i-w/2, a+0.05, f"{a:.1f}", ha="center"); plt.text(i+w/2, b+0.05, f"{b:.1f}", ha="center")
plt.tight_layout(); plt.show()
print("평균 정확성:", {m: round(c,2) for m,c in zip(methods, avg_corr)})
print("평균 완전성:", {m: round(c,2) for m,c in zip(methods, avg_comp)})

### 결과 해설 (왜 이런 결과인가)
- **벡터**: 단일 청크에 답이 다 든 질문(예: 투자 총액 서술)에는 강하지만, multi-hop(투자→CEO)에선 두 사실이 다른 문서라 누락되기 쉽다.
- **그래프**: 연결(투자→CEO, 공통 공급사)은 정확하지만 *금액·연도 서술*은 트리플로 옮길 때 손실될 수 있다.
- **하이브리드**: 연결은 그래프로, 금액/시점 서술은 벡터로 → 보통 **완전성이 가장 높다**. 대신 context가 길어 비용↑.
- 점수는 합성 데이터·judge 변동으로 실행마다 다를 수 있다. **경향(하이브리드 ≥ 그래프·벡터)** 을 본다.

## 8단계 · 🔧 직접 해보기 (3개 이상)

In [ ]:
# 🔧 직접 해보기 — 3개 중 하나씩 골라 주석(#)을 지우고 빈칸을 채워 실행하세요.
# 막히면 힌트 예시를 거의 그대로 따라 해도 됩니다.

# ───────────────────────────────────────────────────────────────
# 🔧 직접 1) 융합 순서를 바꾼 하이브리드 변형 만들기 (그래프를 먼저, 문서를 보조로)
#   힌트: hybrid_rag를 복사해 프롬프트에서 [그래프 관계]를 위에, [관련 문서]를 아래로 두고
#         "연결을 우선 신뢰하고 문서로 보충하라"는 문장을 넣어보세요.
# def hybrid_graph_first(question, hops=2, k=3):
#     g_ctx = subgraph_text(find_entities(question), hops) or "(관련 그래프 관계 없음)"
#     v_ctx = "\n".join(f"- {c}" for c in vector_chunks(question, k))
#     prompt = (f"[그래프 관계 — 우선]\n{g_ctx}\n\n[보조 문서]\n{v_ctx}\n\n"
#               f"질문:{question}\n연결을 우선 신뢰하고 문서로 보충해 답하라.")
#     return ask_claude(prompt)
# print(hybrid_graph_first("그린모빌리티에 투자한 회사의 CEO는?"))

# ───────────────────────────────────────────────────────────────
# 🔧 직접 2) 전역 질문을 QSET에 추가하고 세 방식 점수 다시 그려보기
#   힌트: 아래처럼 (질문, 정답단서) 한 줄을 QSET에 추가한 뒤, 위의 7단계 셀들을 다시 실행하세요.
# QSET.append(("이 생태계의 핵심 공급 허브 기업은?", "정밀소재(노바테크·퀀텀칩 양쪽 공급)"))
#   → 전역 질문에서 graph/hybrid 우위가 더 커지는지 관찰.

# ───────────────────────────────────────────────────────────────
# 🔧 직접 3) "2023년 이후 사건만" 시간순으로 답하게 하기
#   힌트: EVENTS에서 연도가 2023 이상인 것만 골라 timeline을 만들면 됩니다.
# recent = [(yr, desc) for yr, desc in EVENTS if yr >= 2023]
# timeline = "\n".join(f"{yr}년: {desc}" for yr, desc in recent)
# print(ask_claude(f"시간순 사건:\n{timeline}\n\n질문: 2023년 이후 사건을 시간순으로 나열하라."))

print("위 예시 중 하나를 골라 # 을 지우고 실행해보세요.")


## 9단계 · ✅ 검증

### 📌 이 셀이 하는 일 — 자동 점검(검증)
- 그래프가 충분한지, 하이브리드 답에 multi-hop 정답이 들었는지, 하이브리드 완전성이 벡터 이상인지 `assert`로 **자동 확인**합니다.
- 통과하면 `✅ ... 검증 통과`가 찍힙니다. 오류가 나면 위 셀들을 순서대로 다시 실행했는지 보세요.

In [ ]:
assert G.number_of_nodes() >= 6 and G.number_of_edges() >= 6, "그래프가 너무 작음 — 추출 확인"
assert len(EVENTS) >= 4, "시간 이벤트 추출 부족"
# 하이브리드가 multi-hop 정답(김하늘)을 포함해야 함
h_ans = hybrid_rag("그린모빌리티에 투자한 회사의 CEO는 누구인가?")
assert "김하늘" in h_ans, f"하이브리드가 multi-hop 정답을 놓침: {h_ans}"
# 하이브리드 평균 완전성이 단일 벡터 이상이어야(보완 효과)
assert np.mean(scores["hybrid"]["completeness"]) >= np.mean(scores["vector"]["completeness"]) - 0.5, \
    "하이브리드 완전성이 벡터보다 크게 낮음 — 융합 프롬프트 점검"
print("✅ lab5b Hybrid Graph+Vector 검증 통과")

## 확장과제
- **RRF식 융합**: 그래프 경로 점수와 벡터 유사도 점수를 정규화해 가중합(또는 RRF)으로 청크 우선순위를 정한 뒤 컨텍스트 구성.
- **그래프-guided 재질의**: 그래프에서 찾은 이웃 엔티티명을 벡터 쿼리에 추가(쿼리 확장)해 관련 청크 회수율↑.
- **시간 그래프 강화**: 이벤트를 (기업, 연도, 사건) 노드로 모델링하고 `이후/동시` 엣지로 본격 temporal subgraph search 구현.
- 비용 분석: 하이브리드는 context가 길다 — 토큰/비용 대비 완전성 이득을 표로 정리하라.

---
# 파트 3 · Lab 6 — Graph RAG (법률 도메인, 확장/도전 트랙)

## 학습목표
- lab5의 그래프 파이프라인을 **법률 도메인(판례 인용망)**으로 확장한다.
- 인용 **방향그래프**에서 **선례 계보(lineage)**를 추적하고, **쟁점 커뮤니티**를 요약한다.
- 그래프 고유 분석인 **인용 중심성(in-degree / PageRank)**으로 핵심 선례를 식별하고, **벡터 vs 그래프를 judge로 정량 비교·시각화**한다.

> **이 노트북은 자기주도 도전 과제다.** 셀 주석을 충실히 따라 직접 채워가며 진행하라(강사 라이브 진행 의존 최소).
> 사전개념: lab5 골격(추출→정규화→그래프→Local/Global), 인용 네트워크, in-degree/PageRank, map-reduce 요약.
> 비-Claude 사유: networkx(인용 그래프 — Neo4j는 Colab에서 무거움), matplotlib(시각화), sentence-transformers/faiss(벡터 비교군 — Claude는 임베딩 미제공). 데이터는 **가상 판례**(실제 판례·저작권 금지).

## 📖 용어 미니 사전 (먼저 읽으세요)

> 법률 도메인이라 단어가 낯설 수 있지만, 핵심은 lab5와 같습니다 — **점과 선의 지도**예요. 여기선 점=판례/쟁점, 선=인용 관계.

| 용어 | 한 줄 뜻 |
|---|---|
| **지식 그래프 / 노드 / 엣지** | 사실을 점(노드)과 선(엣지)으로 그린 지도. 여기선 점=판례·쟁점·당사자, 선=인용·쟁점 등 관계. |
| **인용(citation)** | 한 판례가 이전 판례(선례)를 **근거로 끌어다 쓰는** 것. "330이 210을 인용". |
| **선례 / 계보(lineage)** | 선례=먼저 나온 판례. 계보=어떤 판례가 직·간접으로 **의존하는 선례 전체의 사슬**. |
| **방향그래프(DiGraph)** | 선에 **방향**이 있는 지도. 인용은 "누가 누구를"이 중요하므로 방향이 필수. |
| **트리플** | (판례, 인용, 선례)처럼 (주어-관계-목적어) 한 묶음. 그래프의 최소 조각. |
| **descendants(후손)** | 한 점에서 화살표를 계속 따라가 **도달하는 모든 점**. 여기선 = 그 판례가 의존하는 선례 계보 전체. |
| **multi-hop / Local / Global search** | 멀티홉=선을 여러 번 갈아타는 질문 / Local=한 판례 주변만 / Global=쟁점 무리별 요약 후 종합. |
| **쟁점 커뮤니티** | 같은 쟁점(개인정보·계약 등)을 다루는 판례들의 **무리**. |
| **in-degree(피인용 수)** | 그 판례가 **다른 판례들에게 인용된 횟수**. 많을수록 자주 참조되는 선례. |
| **PageRank** | "중요한 판례에게 인용받을수록" 높아지는 영향력 점수(단순 횟수보다 '질'을 봄). |
| **judge(채점관)** | 답의 완전성을 Claude가 1~5점으로 매기게 하는 평가. |

## 🧭 왜 "인용 그래프"가 필요할까? (일상 비유로)

논문이나 판결문은 **이전 글을 인용**하며 쌓입니다. 위키백과 한 문서를 읽다가 링크를 타고 들어가고, 또 그 안의 링크를 타고 더 들어가는 것과 같죠.
- "이 판례 **2022가-330**이 직·간접으로 **기대고 있는 선례 전부**를 알려줘"라는 질문은, 330 → (210, 130) → (101, 050) ... 처럼 **링크를 끝까지 따라가야** 답이 나옵니다.
- 비슷한 문장을 찾는 검색(벡터)은 *계보 중간의 판례*가 검색 상위에 안 잡히면 통째로 빠뜨립니다.
- 하지만 **인용을 화살표로 그린 지도**(인용 그래프)가 있으면, 화살표를 자동으로 끝까지 따라가(`descendants`) **계보 전체를 한 번에** 모읍니다.

즉, "**무엇이 무엇에 기대고 있는가**"를 따져야 하는 법률·학술 질문에 그래프가 특히 강합니다. 이번 실습에서 가상 판례로 그 인용 지도를 직접 만들어 봅니다.

## 데이터 · 가상 판례 (합성·내장, 실제 판례 아님, 규모 확대)

인용 관계·쟁점 태그·당사자·법령을 담은 **가상 판례 11건**. 인용망이 충분히 깊도록(여러 단계 계보 + 분기) 구성했다.

### 📌 이 셀이 하는 일 — 가상 판례 데이터 준비
- **실제 판례가 아닌** 가짜 판례 11건을 담습니다(저작권·실제 사건과 무관).
- 각 판례엔 당사자·쟁점·**인용한 선례**·법령이 적혀 있습니다 — 이게 인용 그래프의 재료입니다.
- `11 건 가상 판례 로드`가 보이면 OK.

In [ ]:
CASES = [
    "가상판례 2018가-050: 통신사 데이터 관리 책임. 당사자는 박준과 B통신사. 쟁점은 개인정보보호와 관리책임. 선행 판례 없음. 법령 가상 정보보호법 제15조.",
    "가상판례 2019가-070: 개인정보 위탁처리 책임 범위. 당사자는 서연과 G수탁사. 쟁점은 개인정보보호와 위탁책임. 선행 판례 없음. 법령 가상 정보보호법 제18조.",
    "가상판례 2019가-080: 계약 해지와 위약금. 당사자는 정우와 E쇼핑몰. 쟁점은 계약자유와 위약금 산정. 선행 판례 없음. 법령 가상 민사법 제5조.",
    "가상판례 2020가-101: 개인정보 유출 손해배상. 당사자는 김민과 A통신사. 쟁점은 개인정보보호 의무. 가상판례 2018가-050을 인용. 법령 가상 정보보호법 제20조.",
    "가상판례 2020가-130: 위탁사의 재위탁 통지의무. 당사자는 한울과 H플랫폼. 쟁점은 개인정보보호와 위탁책임. 가상판례 2019가-070을 인용. 법령 가상 정보보호법 제19조.",
    "가상판례 2021가-210: 플랫폼의 개인정보 제3자 제공. 당사자는 이서와 C플랫폼. 쟁점은 개인정보보호와 동의요건. 가상판례 2020가-101과 2018가-050을 인용. 법령 가상 정보보호법 제17조.",
    "가상판례 2021가-250: 자동갱신 고지의무 위반. 당사자는 도윤과 I구독사. 쟁점은 계약자유와 고지의무. 가상판례 2019가-080을 인용. 법령 가상 민사법 제7조.",
    "가상판례 2022가-330: 자동화 의사결정 투명성. 당사자는 최현과 D핀테크. 쟁점은 알고리즘 투명성과 개인정보보호. 가상판례 2021가-210과 2020가-130을 인용. 법령 가상 정보보호법 제22조.",
    "가상판례 2022가-360: 구독 서비스 자동갱신 고지의무. 당사자는 한별과 F구독사. 쟁점은 계약자유와 고지의무. 가상판례 2021가-250과 2019가-080을 인용. 법령 가상 민사법 제8조.",
    "가상판례 2023가-400: 프로파일링 거부권. 당사자는 유나와 J광고사. 쟁점은 알고리즘 투명성과 개인정보보호. 가상판례 2022가-330을 인용. 법령 가상 정보보호법 제23조.",
    "가상판례 2023가-450: 다크패턴과 고지의무. 당사자는 시우와 K커머스. 쟁점은 계약자유와 고지의무. 가상판례 2022가-360과 2021가-250을 인용. 법령 가상 민사법 제9조.",
]
print(len(CASES), "건 가상 판례 로드")

## 1단계 · 도입 — 인용망 질문에 벡터 RAG가 약한 이유

### 개념 복습
"이 판례가 직·간접으로 의존하는 선례 계보 전체는?"은 인용 체인(330→{210,130}→...)이 **여러 판례에 흩어져** 있어 단순 top-k 벡터검색으로 *계보 전체*를 모으기 어렵다. 그래프는 **경로 추적(descendants)**이 자연스럽다. 먼저 벡터의 한계를 관찰한다.

### 📌 이 셀이 하는 일 — 기존 방식(벡터 검색)의 한계 관찰
- 판례 문장들을 숫자로 바꿔 의미가 비슷한 판례를 찾는 검색기를 만들고, **계보 질문**을 던집니다.
- **출력 읽는 법**: `[벡터 RAG]` 답이 계보의 *일부 판례만* 대거나 빠뜨리면, 그게 벡터의 한계입니다(계보 중간이 top-k 밖이면 누락). 뒤에서 그래프가 이걸 어떻게 해결하는지 비교합니다.
> 임베딩·FAISS는 Claude가 제공하지 않아 다른 라이브러리를 씁니다(비교 대상).

In [ ]:
import numpy as np, faiss
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")  # 임베딩=비-Claude
def embed(t): return np.array(embedder.encode(t, normalize_embeddings=True), dtype="float32")
CEMB = embed(CASES); cindex = faiss.IndexFlatIP(CEMB.shape[1]); cindex.add(CEMB)
def vector_chunks(q, k=3):
    _, idx = cindex.search(embed([q]), k); return [CASES[j] for j in idx[0]]
def vector_rag(q, k=3):
    ctx = "\n".join(f"[{i+1}] {c}" for i,c in enumerate(vector_chunks(q,k)))
    return ask_claude(f"근거:\n{ctx}\n\n질문:{q}\n근거만 사용해 답하라.")

CITATION_Q = "가상판례 2022가-330이 직접·간접으로 의존하는 선례 계보를 모두 나열하면?"
print("도전 질문:", CITATION_Q)
print("[벡터 RAG]", vector_rag(CITATION_Q))
# 관찰: 계보의 일부 판례가 top-k 밖이면 누락된다.

## 2단계 · 트리플 추출 (판례-인용-선례 / 쟁점 / 당사자 / 법령)

### 개념 복습
판례 텍스트에서 **(판례, 인용, 선례) / (판례, 쟁점, X) / (판례, 당사자, Y) / (판례, 법령, Z)** 트리플을 Claude tool use로 추출한다. relation을 소수의 통일된 라벨로 강제하는 게 핵심.

### 📌 이 셀이 하는 일 — 판례에서 트리플(관계 조각) 뽑기
- Claude가 각 판례에서 **(판례, 인용, 선례) / (판례, 쟁점, X) / (판례, 당사자, Y) / (판례, 법령, Z)** 형태의 트리플을 뽑습니다.
- `tool use`로 항상 같은 형식(JSON)을 받게 강제하고, `norm_case`로 판례 번호만 깔끔히 통일합니다(예: "가상판례 2022가-330" → "2022가-330").
- **출력 읽는 법**: `추출 N개`와 샘플 목록에서 `2022가-330 - 인용 -> 2021가-210` 같은 줄이 보이면 성공입니다.

In [ ]:
# 정확도가 더 필요하면 이 judge/추출 셀의 모델을 MODEL_JUDGE = "claude-opus-4-8" 로 한 줄 상향 가능(기본은 비용 위해 sonnet 4.6).
TRIPLE_TOOL = [{"name":"emit_triples","description":"판례 텍스트에서 트리플 추출",
  "input_schema":{"type":"object","properties":{"triples":{"type":"array","items":{"type":"object",
    "properties":{"head":{"type":"string"},"relation":{"type":"string"},"tail":{"type":"string"}},
    "required":["head","relation","tail"]}}},"required":["triples"]}}]

def extract_triples(text):
    msg = client.messages.create(model=MODEL, max_tokens=1200,
        tools=TRIPLE_TOOL, tool_choice={"type":"tool","name":"emit_triples"},
        messages=[{"role":"user","content":
            "다음 가상 판례에서 트리플을 추출하라. relation은 인용/쟁점/당사자/법령 중 하나로 통일. "
            "head는 항상 판례 번호(예: 2022가-330), 인용의 tail도 판례 번호여야 한다.\n"+text}])
    for b in msg.content:
        if b.type=="tool_use": return b.input["triples"]
    return []

# 정규화: 판례 번호만 깔끔히 (예: "가상판례 2022가-330" -> "2022가-330")
def norm_case(s):
    m = re.search(r"(20\d{2}가-\d+)", s)
    return m.group(1) if m else s.strip()

all_triples = []
for c in CASES:                          # 예상 호출: 판례 수(11)
    all_triples += extract_triples(c)
for t in all_triples:
    if t["relation"].strip()=="인용":
        t["head"]=norm_case(t["head"]); t["tail"]=norm_case(t["tail"])
    else:
        t["head"]=norm_case(t["head"])
print(f"추출 {len(all_triples)}개 (샘플 12):")
for t in all_triples[:12]:
    print("  ", t["head"], "-", t["relation"], "->", t["tail"])

## 3단계 · 인용 방향그래프 + 쟁점/법령 노드 구축 + 시각화

### 개념 복습
전체 그래프 `G`(인용+쟁점+당사자+법령)와, 중심성 분석용 **인용 전용 서브그래프 `CITES`**를 따로 만든다. 인용은 방향이 핵심(누가 누구를 인용)이다.

### 📌 이 셀이 하는 일 — 인용 지도(방향그래프) 만들고 그림 그리기
- 전체 관계는 그래프 `G`에, 그중 **인용만** 뽑아 인용 전용 그래프 `CITES`에 따로 담습니다(중심성 분석용).
- **🖼 그림에서 볼 점**: 동그라미=판례·쟁점·당사자·법령, 화살표=관계(인용/쟁점 등). **인용 화살표가 한 방향으로 흐르는 사슬**(예: 450→360→250)을 눈으로 따라가 보세요. 화살표가 많이 **들어오는** 판례가 자주 인용된 핵심 선례입니다.

In [ ]:
import networkx as nx, matplotlib.pyplot as plt
G = nx.DiGraph()
for t in all_triples:
    G.add_edge(t["head"].strip(), t["tail"].strip(), relation=t["relation"].strip())
CITES = nx.DiGraph()
for u,v,d in G.edges(data=True):
    if "인용" in d["relation"]:
        CITES.add_edge(u, v)
print(f"전체: 노드 {G.number_of_nodes()} 엣지 {G.number_of_edges()} | 인용그래프: 노드 {CITES.number_of_nodes()} 엣지 {CITES.number_of_edges()}")

plt.figure(figsize=(11,7))
pos = nx.spring_layout(G, seed=7, k=1.0)
nx.draw(G, pos, with_labels=True, node_color="#ffe0cc", node_size=1200, font_size=7)
nx.draw_networkx_edge_labels(G, pos, edge_labels=nx.get_edge_attributes(G,"relation"), font_size=6)
plt.title("Legal Citation Graph (synthetic)"); plt.axis("off"); plt.show()

## 4단계 · Local search — 인용 체인(선례 계보) 추적

### 개념 복습
인용 그래프에서 한 판례가 **도달하는 모든 후손 노드(descendants)** = 그 판례가 직·간접으로 의존하는 선례 계보. `nx.descendants`로 경로 전체를 한 번에 모은다 — 벡터가 어려워하던 바로 그 작업.

### 📌 이 셀이 하는 일 — Local search: 인용 계보를 자동으로 끝까지 추적
- `nx.descendants(CITES, 판례)`는 그 판례에서 인용 화살표를 따라 **도달하는 모든 선례**를 한 번에 모읍니다 = 계보 전체.
- 이게 벡터가 어려워하던 바로 그 작업입니다.
- **출력 읽는 법**: `2022가-330 선례 계보: [...]`에 여러 단계 판례가 빠짐없이 들어오면, 그래프가 계보를 정확히 추적한 것입니다.

In [ ]:
def citation_lineage(case):
    case = case if case in CITES else None
    if case is None: return []
    return sorted(nx.descendants(CITES, case))

def local_search(question, case):
    lineage = citation_lineage(case)
    edges = [f"{u} -인용-> {v}" for u,v in CITES.edges() if u==case or u in lineage]
    ctx = "\n".join(edges) or "인용 관계 없음"
    return ask_claude(f"인용 관계:\n{ctx}\n\n질문:{question}\n관계만 사용해 계보를 설명하라.")

print("2022가-330 선례 계보:", citation_lineage("2022가-330"))
print("[Graph Local]", local_search(CITATION_Q, "2022가-330"))

## 5단계 · Global search — 쟁점 커뮤니티 요약 (map-reduce)

### 개념 복습
쟁점(relation=쟁점) 태그로 판례를 묶어 **쟁점 커뮤니티**를 만들고, 각 쟁점군을 Claude로 요약(map)한 뒤 전역 질문에 종합(reduce)한다.

### 📌 이 셀이 하는 일 — Global search: 쟁점별로 묶어 요약 후 종합
- 같은 **쟁점**(개인정보·계약 등)을 다루는 판례들을 무리(커뮤니티)로 묶고, 각 무리를 Claude로 **요약(map)**한 뒤 전체를 **종합(reduce)**합니다.
- **출력 읽는 법**: `쟁점 클러스터`로 어떤 판례들이 같은 쟁점에 묶였는지 보이고, `[Graph Global]`에 쟁점군 전반의 **판단 경향**을 정리한 답이 나오면 성공입니다.

In [ ]:
def issue_clusters():
    clusters = {}
    for u,v,d in G.edges(data=True):
        if "쟁점" in d["relation"]:
            clusters.setdefault(v.strip(), []).append(u.strip())
    return clusters

def global_search(question):
    parts = []
    for issue, cases in issue_clusters().items():     # map
        parts.append(ask_claude(f"쟁점 '{issue}' 관련 가상판례 {sorted(set(cases))}의 공통 판단 경향을 1-2문장으로 요약."))
    joined = "\n".join(f"- {p}" for p in parts)
    return ask_claude(f"쟁점별 요약:\n{joined}\n\n전역 질문:{question}\n종합해 답하라.")  # reduce

print("쟁점 클러스터:", {k: sorted(set(v)) for k,v in issue_clusters().items()})
print("[Graph Global]", global_search("개인정보보호 관련 가상판례군의 전반적 판단 경향은?"))

## 6단계 · 인용 중심성 분석 (in-degree vs PageRank)

### 개념 복습
- **in-degree(피인용 수)**: 많이 인용된 = 자주 참조되는 선례.
- **PageRank**: "중요한 판례에게 인용받은" 판례에 더 큰 점수 → 영향력의 질을 반영.
- 둘의 최상위가 다를 수 있다(많이 인용 ≠ 영향력 큰 곳에서 인용). 막대그래프로 비교한다.

### 📌 이 셀이 하는 일 — 핵심 선례 찾기 (in-degree vs PageRank)
- **in-degree** = 단순히 많이 인용된 횟수. **PageRank** = "영향력 큰 판례에게 인용받을수록" 높은 점수(질을 봄).
- 둘의 1등이 다를 수 있어, 막대그래프로 나란히 비교합니다.
- **🖼 그림에서 볼 점**: 판례마다 막대 2개(피인용 수 vs PageRank). **두 막대의 키 순서가 뒤바뀌는 판례**가 흥미로운 지점입니다 — "많이 인용 ≠ 영향력 큰 곳에서 인용"을 보여줍니다.

In [ ]:
indeg = dict(CITES.in_degree())
pr = nx.pagerank(CITES) if CITES.number_of_edges() else {}
top_indeg = sorted(indeg.items(), key=lambda x:x[1], reverse=True)
top_pr = sorted(pr.items(), key=lambda x:x[1], reverse=True)
print("in-degree(피인용):", top_indeg[:5])
print("PageRank:", [(k, round(v,3)) for k,v in top_pr[:5]])

nodes = [k for k,_ in top_pr]
x = np.arange(len(nodes)); w=0.4
plt.figure(figsize=(10,4))
plt.bar(x-w/2, [indeg.get(n,0) for n in nodes], w, label="in-degree(피인용 수)")
plt.bar(x+w/2, [pr.get(n,0)*max(1,sum(indeg.values())) for n in nodes], w, label="PageRank(스케일)")
plt.xticks(x, nodes, rotation=20); plt.title("핵심 선례 — in-degree vs PageRank"); plt.legend()
plt.tight_layout(); plt.show()
print("\n해석: 피인용 최다 선례와 PageRank 최상위가 다르면, 후자는 '영향력 큰 판례들에게' 인용받았음을 뜻한다.")

## 7단계 · 벡터 vs 그래프 정량 비교 (judge 채점 + 시각화)

### 개념 복습
인용망/계보/전역 질문셋을 두 방식으로 답하고 judge로 **완전성(계보·근거 누락 없이)**을 채점, 막대그래프로 비교한다.

### 📌 이 셀이 하는 일 — 벡터 vs 그래프를 점수로 비교
- 계보·전역 질문 3개를 두 방식으로 답하게 하고, Claude **채점관**이 **완전성**(계보·근거를 빠짐없이 담았는가)을 1~5점으로 매깁니다.
- **출력 읽는 법**: 질문별 두 방식 점수가 표로 나옵니다. 계보 질문에서 보통 **그래프가 높습니다**.

In [ ]:
# 정확도가 더 필요하면 이 judge/추출 셀의 모델을 MODEL_JUDGE = "claude-opus-4-8" 로 한 줄 상향 가능(기본은 비용 위해 sonnet 4.6).
JUDGE_TOOL = [{"name":"judge","description":"답변 채점",
  "input_schema":{"type":"object","properties":{"reasoning":{"type":"string"},
    "completeness":{"type":"integer","description":"1-5 완전성(계보/근거 누락 없이)"}},
    "required":["reasoning","completeness"]}}]
def judge_answer(question, answer, gold):
    msg = client.messages.create(model=MODEL, max_tokens=450,
        tools=JUDGE_TOOL, tool_choice={"type":"tool","name":"judge"},
        messages=[{"role":"user","content":
            f"질문:{question}\n참고 정답 단서:{gold}\n답변:{answer}\n완전성 1-5(reasoning 먼저)."}])
    for b in msg.content:
        if b.type=="tool_use": return b.input.get("completeness", 0) # Fixed: Safely get 'completeness'
    return 0

# 예상 호출: 3질문 x (2답변+2judge)=12건 (+ global_search 내부 map 호출).
QSET = [
    ("2022가-330이 직접·간접 의존하는 선례 계보 전체는?", "210,130,101,050 등 다단 계보", "local", "2022가-330"),
    ("2023가-450이 의존하는 선례 계보 전체는?", "360,250,080 등", "local", "2023가-450"),
    ("개인정보보호 쟁점 판례군의 전반적 판단 경향은?", "개인정보보호 의무 강화 경향 종합", "global", None),
]
v_scores, g_scores, labels = [], [], []
for q, gold, mode, case in QSET:
    v = vector_rag(q)
    g = global_search(q) if mode=="global" else local_search(q, case)
    v_scores.append(judge_answer(q, v, gold)); g_scores.append(judge_answer(q, g, gold))
    labels.append(q[:16])
print(f"{'질문':<18}{'벡터':<6}{'그래프'}")
for l,v,g in zip(labels, v_scores, g_scores):
    print(f"{l:<18}{v:<6}{g}")

### 📌 이 셀이 하는 일 — 비교 결과를 막대그래프로 그리기
- 질문별 완전성 점수를 막대그래프로 그립니다.
- **🖼 그림에서 볼 점**: 질문마다 벡터·그래프 막대가 나란히 섭니다. **그래프 막대가 더 높은 질문**이 "인용망이라서 그래프가 이긴" 질문입니다. 맨 아래에 평균도 찍힙니다.

In [ ]:
x = np.arange(len(labels)); w=0.38
plt.figure(figsize=(9,4))
plt.bar(x-w/2, v_scores, w, label="벡터 RAG")
plt.bar(x+w/2, g_scores, w, label="Graph RAG")
plt.xticks(x, labels, rotation=15); plt.ylim(0,5.3); plt.ylabel("완전성(1-5)")
plt.title("Vector vs Graph RAG — 인용망 질문 완전성"); plt.legend()
plt.tight_layout(); plt.show()
print(f"평균 완전성  벡터={np.mean(v_scores):.2f}  그래프={np.mean(g_scores):.2f}")

### 결과 해설
- 계보 추적 질문에서 **그래프가 완전성에서 앞선다**: `descendants`로 다단 계보를 빠짐없이 모으기 때문.
- 벡터는 top-k 밖의 선례를 놓치기 쉽다.
- 전역(쟁점 경향) 질문도 그래프의 커뮤니티 요약이 누락을 줄인다.

## 8단계 · 시간적 추론 — 판례 선후 관계 (확장)

### 개념 복습
판례 번호의 **연도**를 파싱해 선후를 그래프에 반영한다. "최신 판례가 의존하는 가장 오래된 선례는?" 같은 **시간 추론 질문**을 계보 + 연도로 답한다.

### 📌 이 셀이 하는 일 — 시간 추론: 계보에서 가장 오래된 선례 찾기
- 판례 번호에서 **연도**를 뽑아, 한 판례의 계보 중 **가장 오래된 뿌리 선례**를 찾습니다.
- **출력 읽는 법**: `... → 가장 오래된 선례: (연도, 판례)` 형태로, 계보의 시작점이 보입니다.

In [ ]:
def case_year(c):
    m = re.search(r"(20\d{2})", c); return int(m.group(1)) if m else None

def oldest_in_lineage(case):
    line = citation_lineage(case) + [case]
    yrs = [(case_year(c), c) for c in line if case_year(c)]
    return min(yrs) if yrs else None

for tgt in ["2022가-330", "2023가-450"]:
    line = citation_lineage(tgt)
    oldest = oldest_in_lineage(tgt)
    print(f"{tgt} 계보 {line} → 가장 오래된 선례: {oldest}")
# 🔧 직접: '2021년 이후 판례만'으로 계보를 필터링해 최근 흐름만 추적해보라.

## 9단계 · 🔧 직접 해보기 (3개 이상)

In [ ]:
# 🔧 직접 해보기 — 3개 중 하나씩 골라 주석(#)을 지우고 빈칸을 채워 실행하세요.
# 막히면 힌트 예시를 거의 그대로 따라 해도 됩니다.

# ───────────────────────────────────────────────────────────────
# 🔧 직접 1) in-degree 1등과 PageRank 1등 판례가 왜 다른지 확인하고, 인용 하나 추가해 보기
#   힌트: 먼저 두 지표의 1등을 출력해 비교한 뒤, 영향력 큰 판례가 어떤 선례를 인용하도록 엣지를 추가합니다.
# print("in-degree 1등:", max(indeg, key=indeg.get), " / PageRank 1등:", max(pr, key=pr.get))
# CITES.add_edge("2023가-450", "2020가-101")   # 예시 인용 추가
# import networkx as nx
# print("추가 후 PageRank 1등:", max(nx.pagerank(CITES), key=nx.pagerank(CITES).get))

# ───────────────────────────────────────────────────────────────
# 🔧 직접 2) 새 판례를 추가하고 계보가 갱신되는지 확인하기
#   힌트: 기존 판례를 인용하는 새 판례 문장을 CASES에 추가한 뒤, 2단계(추출)~3단계(그래프) 셀을
#         다시 실행하고 citation_lineage로 새 계보를 확인합니다.
# CASES.append("가상판례 2024가-500: 다크패턴 손해배상. 당사자는 하준과 L몰. 쟁점은 계약자유와 고지의무. 가상판례 2023가-450을 인용. 법령 가상 민사법 제10조.")
#   → 위 줄을 추가하고 2·3단계 셀 재실행 후:  print(citation_lineage("2024가-500"))

# ───────────────────────────────────────────────────────────────
# 🔧 직접 3) 질문에 따라 Local/Global을 자동으로 고르는 router 만들기
#   힌트: "경향/판례군/전반" 같은 큰 단어가 있으면 global, 아니면 local.
# def route(q):
#     return "global" if any(w in q for w in ["경향", "판례군", "전반"]) else "local"
# print(route("개인정보 판례군의 경향은?"), "/", route("2022가-330의 선례 계보는?"))

print("위 예시 중 하나를 골라 # 을 지우고 실행해보세요.")


## 10단계 · ✅ 검증

### 📌 이 셀이 하는 일 — 자동 점검(검증)
- 인용 그래프가 충분한지, 2022가-330의 계보에 직접 선례(2021가-210)와 간접 선례가 들어 있는지 등을 `assert`로 **자동 확인**합니다.
- 통과하면 `✅ ... 검증 통과`와 계보가 찍힙니다. 오류 시 위 셀들을 순서대로 다시 실행하세요.

In [ ]:
assert CITES.number_of_edges() >= 8, f"인용 관계가 적음({CITES.number_of_edges()}) — 추출/정규화 확인"
assert CITES.number_of_nodes() >= 8, f"인용 노드가 적음({CITES.number_of_nodes()})"
# 계보: 2022가-330은 2021가-210을 통해 2020가-101, 2018가-050까지 닿아야 함
lineage = citation_lineage("2022가-330")
assert "2021가-210" in lineage, f"직접 인용 선례 누락: {lineage}"
assert any(x in lineage for x in ["2020가-101","2018가-050","2020가-130"]), f"간접 선례 계보 누락: {lineage}"
# 중심성 계산 정상
assert len(pr) == CITES.number_of_nodes() and max(pr.values()) > 0, "PageRank 이상"
# 정량 비교에서 그래프 평균 완전성이 벡터 이상(경향)
assert np.mean(g_scores) >= np.mean(v_scores) - 0.5, "그래프가 벡터보다 크게 낮음 — 점검"
print("✅ lab6 Graph RAG(법률) 검증 통과 | 2022가-330 계보:", lineage)

## 확장과제 (자기주도)
- **도메인 일반화**: lab5(금융)·lab6(법률)의 `extract_triples`/그래프 코드를 하나의 파이프라인으로 합쳐 두 데이터를 같은 함수로 처리.
- **하이브리드(→ lab5b)**: 인용 서브그래프 + 판례 원문 청크를 함께 컨텍스트로 줘 답변 완전성 비교.
- **시간 그래프 강화**: 연도 엣지(`이후`)를 명시적으로 추가해 temporal subgraph search 구현.
- 인용 중심성 상위 판례를 Local search의 우선 컨텍스트로 가중.